In [ ]:
!pip install nltk

In [ ]:
import nltk

review = "The phone has an amazing camera but terrible battery life."

aspects = {
    "camera": ["camera"],
    "battery": ["battery", "battery life"]
}

positive_words = ["amazing", "excellent", "great", "good"]
negative_words = ["terrible", "bad", "poor", "worst"]

review = review.lower()

for aspect, keywords in aspects.items():

    sentiment = "NEUTRAL"

    for keyword in keywords:

        if keyword in review:

            for word in positive_words:
                if word in review:
                    sentiment = "POSITIVE"

            for word in negative_words:
                if word in review:
                    sentiment = "NEGATIVE"

    print(aspect, ":", sentiment)

## **Practical 2: spaCy + Aspect Extraction**

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "The phone has an amazing camera but terrible battery life."

doc = nlp(text)

for token in doc:

    if token.pos_ == "NOUN":

        print(token.text
              )

# **Practical 3: ABSA using Transformer**

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

review = """
The camera is amazing but the battery life is terrible.
"""

camera_result = sentiment("The camera is amazing")[0]

battery_result = sentiment("The battery life is terrible")[0]

print("Camera :", camera_result)
print("Battery:", battery_result)

In [ ]:
!pip install nltk scikit-learn pandas seaborn matplotlib joblib

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string
import joblib

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
data = {
    "text": [
        "Python is great for machine learning and AI development",
        "The stock market crashed after poor earnings reports",
        "Manchester United won the Premier League championship",
        "New deep learning model beats GPT 4 on benchmarks",
        "Federal Reserve raises interest rates by 25 basis points",
        "LeBron James scores 40 points in NBA Finals game 7",
        "Tesla launches new self driving software",
        "Gold prices rise amid economic uncertainty",
        "India wins cricket world cup",
        "OpenAI releases new language model"
    ],

    "label": [
        "tech",
        "finance",
        "sports",
        "tech",
        "finance",
        "sports",
        "tech",
        "finance",
        "sports",
        "tech"
    ]
}

df = pd.DataFrame(data)

df.head()

In [ ]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

In [ ]:
stop_words

In [ ]:
def preprocess(text):

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', '', text)

    text = re.sub(r'@\w+|#\w+', '', text)

    text = re.sub(r'\d+', 'NUM', text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [ ]:
df["clean_text"] = df["text"].apply(preprocess)

df[["text","clean_text"]]

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=1000
)

X = tfidf.fit_transform(df["clean_text"])

print(X.shape)

In [ ]:
print(tfidf.get_feature_names_out())

In [ ]:
tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

In [ ]:
encoder = LabelEncoder()

y = encoder.fit_transform(df["label"])

print(y)

In [ ]:
encoder.classes_

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ** Train Multiple Models**
**1. Naive Bayes**

In [ ]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

pred_nb = nb.predict(X_test)

print(classification_report(y_test, pred_nb))

**2. Logistic Regression**

In [ ]:
lr = LogisticRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print(classification_report(y_test, pred_lr))

**3. Linear SVM**

In [ ]:
svm = LinearSVC()

svm.fit(X_train, y_train)

pred_svm = svm.predict(X_test)

print(classification_report(y_test, pred_svm))

In [ ]:
# Compare the model using the cross validation
models = {

    "Naive Bayes": MultinomialNB(),

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Linear SVM":
        LinearSVC()
}

In [ ]:
for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring='f1_macro'
    )

    print(
        f"{name}: "
        f"{scores.mean():.3f}"
    )

In [ ]:
cm = confusion_matrix(y_test, pred_lr)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
new_text = [
    "OpenAI launches powerful GPT model"
]

In [ ]:
clean_text = preprocess(new_text[0])

vector = tfidf.transform([clean_text])

prediction = lr.predict(vector)

label = encoder.inverse_transform(prediction)

print(label[0])

In [ ]:
samples = [

    "India wins cricket series",

    "Stock market falls sharply",

    "Google launches AI model"
]

In [ ]:
for sentence in samples:

    clean = preprocess(sentence)

    vector = tfidf.transform([clean])

    pred = lr.predict(vector)

    label = encoder.inverse_transform(pred)

    print(sentence, "->", label[0])